# Fusion Retrieval（融合检索：向量检索 + BM25）

## 核心想法

- **向量检索（dense / semantic）**：擅长找“语义相近”的段落，但不擅长精确匹配专有名词、缩写、数字、关键短语。
- **BM25（sparse / keyword）**：擅长关键词精确匹配，但对同义改写/语义泛化不敏感。

**融合检索（Fusion / Hybrid Retrieval）** 就是把两种信号合并，让检索更稳。

下面我们会：

1. 用 PDF 构建 chunks
2. 建 FAISS（向量库）
3. 建 BM25（关键词索引）
4. 对同一个 query 同时打分并归一化，然后用 `alpha` 做加权融合

<div style="text-align: center;">

<img src="../images/fusion_retrieval.svg" alt="Fusion Retrieval" style="width:100%; height:auto;">
</div>


In [1]:
from __future__ import annotations

import os
import re
import sys
from pathlib import Path
from typing import List

import numpy as np
import requests
from dotenv import load_dotenv
from pypdf import PdfReader
from rank_bm25 import BM25Okapi

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py`
sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from helper_functions import show_context

/tmp/ipykernel_4163638/2775394911.py:24: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) 下载 PDF 并抽取为纯文本

我们使用示例文件 `Understanding_Climate_Change.pdf`，把每页文本提取出来并拼接成一个大字符串 `content`。

PDF 抽取不是重点；我们只需要得到可用的正文文本。

In [2]:
os.environ["HTTP_PROXY"] = os.getenv("HTTP_PROXY", "http://127.0.0.1:7890")
os.environ["HTTPS_PROXY"] = os.getenv("HTTPS_PROXY", "http://127.0.0.1:7890")
os.environ["ALL_PROXY"] = os.getenv("ALL_PROXY", "http://127.0.0.1:7890")

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

PDF_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/Understanding_Climate_Change.pdf"
PDF_PATH = DATA_DIR / "Understanding_Climate_Change.pdf"

if not PDF_PATH.exists():
    resp = requests.get(
        PDF_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    resp.raise_for_status()
    PDF_PATH.write_bytes(resp.content)

reader = PdfReader(str(PDF_PATH))
pages_text = [(p.extract_text() or "") for p in reader.pages]
content = "\n\n".join(pages_text)
content = content.replace("\t", " ")

assert len(content.strip()) > 0, "PDF 提取到的文本为空（pypdf.extract_text 可能失败）"
print("Pages:", len(reader.pages))
print("Chars:", len(content))

Pages: 33
Chars: 72592


## 2) 切分 chunks，并同时建立两套索引

- **向量索引（FAISS）**：用于语义相似检索
- **BM25 索引**：用于关键词检索

两者都基于同一批 chunks，方便后面做融合。

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)

raw_doc = Document(page_content=content, metadata={"source": str(PDF_PATH)})
docs: List[Document] = text_splitter.split_documents([raw_doc])

# 给每个 chunk 一个稳定的 id，后面融合时用它对齐两种分数
for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)


def tokenize(text: str) -> List[str]:
    # 简单、可解释的 tokenizer：只保留字母数字
    return re.findall(r"[A-Za-z0-9]+", text.lower())


tokenized_docs = [tokenize(d.page_content) for d in docs]
bm25 = BM25Okapi(tokenized_docs)

print("Num chunks:", len(docs))

Num chunks: 97


## 3) 定义融合检索：两种打分 → 归一化 → 加权融合

我们对同一个 query 做两次检索打分：

- BM25：分数越大越相关
- 向量检索：FAISS 返回的是“距离”（越小越相似），所以要先转成“相似度方向”的分数，再做归一化

最后用：

$$
score = \alpha \cdot score_{vector} + (1-\alpha) \cdot score_{bm25}
$$

其中 `alpha` 越大越偏向语义检索，越小越偏向关键词检索。

In [5]:
def _minmax(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = x.astype(float)
    return (x - x.min()) / (x.max() - x.min() + eps)


def fusion_retrieval(
    vectorstore: FAISS,
    bm25: BM25Okapi,
    query: str,
    k: int = 5,
    alpha: float = 0.5,
) -> List[Document]:
    """融合检索：BM25（关键词）+ 向量（语义）。

    返回：按融合分数排序后的 top-k `Document`（仍然是原始 chunk）。
    """

    # 1) BM25：对所有 docs 打分（长度 = len(docs)）
    bm25_scores = np.array(bm25.get_scores(tokenize(query)))
    bm25_scores_norm = _minmax(bm25_scores)

    # 2) 向量检索：拿到每个 doc 的“距离”
    # 注意：FAISS 的 score 通常是距离（越小越相似）。
    vector_results = vectorstore.similarity_search_with_score(query, k=len(docs))

    # 建一个 chunk_id -> distance 的映射，便于对齐到 bm25_scores（按 docs 顺序）
    dist_by_id = {d.metadata["chunk_id"]: float(dist) for d, dist in vector_results}
    vector_dists = np.array([dist_by_id[i] for i in range(len(docs))])

    # 转成“越大越相似”的方向，然后归一化到 [0, 1]
    vector_sim_norm = 1 - _minmax(vector_dists)

    # 3) 加权融合
    combined = alpha * vector_sim_norm + (1 - alpha) * bm25_scores_norm

    # 4) 取 top-k
    top_idx = np.argsort(combined)[::-1][:k]
    return [docs[i] for i in top_idx]


def bm25_only(query: str, k: int = 5) -> List[Document]:
    scores = np.array(bm25.get_scores(tokenize(query)))
    top_idx = np.argsort(scores)[::-1][:k]
    return [docs[i] for i in top_idx]


def vector_only(query: str, k: int = 5) -> List[Document]:
    return [d for d, _ in vectorstore.similarity_search_with_score(query, k=k)]

## 4) 用例：同一个 query，对比三种检索结果

我们对比：

- 只用向量检索（semantic）
- 只用 BM25（keyword）
- 融合检索（fusion）

你可以调 `alpha` 观察输出如何偏向语义/关键词。

In [6]:
query = "What are the impacts of climate change on the environment?"

print("=== Query ===\n")
print(query)

print("\n=== Vector only ===\n")
vec_docs = vector_only(query, k=5)
show_context([d.page_content for d in vec_docs])

print("\n=== BM25 only ===\n")
bm25_docs = bm25_only(query, k=5)
show_context([d.page_content for d in bm25_docs])

print("\n=== Fusion (alpha=0.5) ===\n")
fusion_docs = fusion_retrieval(vectorstore, bm25, query, k=5, alpha=0.5)
show_context([d.page_content for d in fusion_docs])

=== Query ===

What are the impacts of climate change on the environment?

=== Vector only ===

Context 1:
Climate change is altering terrestrial ecosystems by shifting habitat ranges, changing species 
distributions, and impacting ecosystem functions. Forests, grasslands, and deserts are 
experiencing shifts in plant and animal species composition. These changes can lead to a loss 
of biodiversity and disrupt ecological balance. 
Marine Ecosystems 
Marine ecosystems are highly vulnerable to climate change. Rising sea temperatures, ocean 
acidification, and changing currents affect marine biodiversity, from coral reefs to deep-sea 
habitats. Species migration and changes in reproductive cycles can disrupt marine food webs 
and fisheries.

Context 2:
development of eco-friendly fertilizers and farming techniques is essential for reducing the 
agricultural sector's carbon footprint. 
Chapter 3: Effects of Climate Change 
The effects of climate change are already being felt around the wor